In [1]:
import warnings
warnings.filterwarnings('ignore')

In [50]:
import os
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import pyarrow.parquet as pq

from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score, roc_auc_score

### Обучаю линейную регрессию с помощью PyTorch

In [3]:
# задаем параметры
BATCH_SIZE = 1000
L_RATE = 0.01
N_EPOCHS = 20

In [4]:
data_train = pd.read_parquet('data_train_std.pq')

In [5]:
data_train.shape

(2400000, 388)

In [6]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(data_train['flag']),
    y=data_train['flag']
)
class_weights

array([ 0.51839302, 14.09211546])

In [7]:
X_train, y_train = data_train.drop(columns=['id', 'flag']), data_train[['flag']]

In [8]:
X_train.shape, y_train.shape

((2400000, 386), (2400000, 1))

In [9]:
del data_train
gc.collect()

0

In [10]:
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)
    
    def __len__(self):
        return self.X.shape[0]
        
    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [11]:
train_dataset = MyDataset(X_train, y_train)

In [12]:
train_dataloader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,          
    num_workers=4,
    drop_last=True
)

In [13]:
pos_weight = torch.tensor([class_weights[1] / class_weights[0]])
pos_weight

tensor([27.1842], dtype=torch.float64)

In [14]:
model = nn.Linear(X_train.shape[1], 1)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.SGD(model.parameters(), lr=L_RATE)

In [15]:
epoch_losses = {}
epoch_accuracies = {}
epoch_roc_aucs = {}

In [16]:
%%time
for epoch in range(N_EPOCHS):
    epoch_loss = 0
    all_logits = []
    all_labels = []

    model.train()
    
    for xb, yb in train_dataloader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        epoch_loss += loss.item()
        all_logits += logits.tolist()
        all_labels += yb.tolist()
        loss.backward()
        optimizer.step()

    epoch_loss = epoch_loss / len(train_dataloader)
    
    all_logits = torch.tensor(all_logits)
    all_labels = torch.tensor(all_labels)

    y_pred_binary = torch.round(torch.sigmoid(all_logits)).cpu().numpy()
    y_true = all_labels.cpu().numpy()

    epoch_accuracy = accuracy_score(y_true, y_pred_binary)
    epoch_roc_auc = roc_auc_score(y_true, all_logits.cpu().numpy())

    print(f'Epoch {epoch+1:02}, Epoch loss = {epoch_loss:.2f}, Epoch accuracy = {epoch_accuracy:.3f}, Epoch ROC AUC = {epoch_roc_auc:.3f}')

    epoch_losses[epoch + 1] = epoch_loss
    epoch_accuracies[epoch + 1] = epoch_accuracy
    epoch_roc_aucs[epoch + 1] = epoch_roc_auc

Epoch 01, Epoch loss = 1.20, Epoch accuracy = 0.633, Epoch ROC AUC = 0.720
Epoch 02, Epoch loss = 1.19, Epoch accuracy = 0.655, Epoch ROC AUC = 0.727
Epoch 03, Epoch loss = 1.19, Epoch accuracy = 0.655, Epoch ROC AUC = 0.728
Epoch 04, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.729
Epoch 05, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.729
Epoch 06, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.730
Epoch 07, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.730
Epoch 08, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.731
Epoch 09, Epoch loss = 1.18, Epoch accuracy = 0.656, Epoch ROC AUC = 0.731
Epoch 10, Epoch loss = 1.18, Epoch accuracy = 0.656, Epoch ROC AUC = 0.731
Epoch 11, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.731
Epoch 12, Epoch loss = 1.18, Epoch accuracy = 0.656, Epoch ROC AUC = 0.731
Epoch 13, Epoch loss = 1.18, Epoch accuracy = 0.655, Epoch ROC AUC = 0.731
Epoch 14, Epoch loss = 1.

### Обучаю линейную регрессию SGDClassifier с помощью Scikit-learn

In [18]:
# задаем параметры
LOSS = 'log_loss'
PENALTY = 'l2'
ALPHA = 1e-4
SEED = 42
CLASSES = [0, 1]

In [19]:
clf = SGDClassifier(loss=LOSS, penalty=PENALTY, alpha=ALPHA, random_state=SEED)

In [20]:
reader = pq.ParquetFile('data_train_std.pq')

# Общее число строк в файле
total_rows = reader.metadata.num_rows
total_rows

2400000

In [27]:
# Число строк в i‑й row group
for i in range(reader.num_row_groups):
    num_rows_in_rg = reader.metadata.row_group(i).num_rows
    print(f'Row group {i} содержит {num_rows_in_rg} строк')

Row group 0 содержит 1048576 строк
Row group 1 содержит 1048576 строк
Row group 2 содержит 302848 строк


In [30]:
train_cols = [col for col in reader.schema.names if col not in ['id', 'flag']]
train_cols

['pre_pterm_14_std',
 'pre_pterm_15_std',
 'pre_pterm_16_std',
 'pre_pterm_17_std',
 'pre_fterm_1_std',
 'pre_fterm_2_std',
 'pre_fterm_3_std',
 'pre_fterm_4_std',
 'pre_fterm_5_std',
 'pre_fterm_6_std',
 'pre_fterm_7_std',
 'pre_fterm_8_std',
 'pre_fterm_9_std',
 'pre_fterm_10_std',
 'pre_fterm_11_std',
 'pre_fterm_12_std',
 'pre_fterm_13_std',
 'pre_fterm_14_std',
 'pre_fterm_15_std',
 'pre_fterm_16_std',
 'pre_till_pclose_1_std',
 'pre_till_pclose_2_std',
 'pre_till_pclose_3_std',
 'pre_till_pclose_4_std',
 'pre_till_pclose_5_std',
 'pre_till_pclose_6_std',
 'pre_till_pclose_7_std',
 'pre_till_pclose_8_std',
 'pre_till_pclose_9_std',
 'pre_till_pclose_10_std',
 'pre_till_pclose_11_std',
 'pre_till_pclose_12_std',
 'pre_till_pclose_13_std',
 'pre_till_pclose_14_std',
 'pre_till_pclose_15_std',
 'pre_till_pclose_16_std',
 'pre_till_fclose_1_std',
 'pre_till_fclose_2_std',
 'pre_till_fclose_3_std',
 'pre_till_fclose_4_std',
 'pre_till_fclose_5_std',
 'pre_till_fclose_6_std',
 'pre_till

In [31]:
first_call = True

for i in range(reader.num_row_groups):
    table = reader.read_row_group(i)
    df_batch = table.to_pandas()
    
    del table
    gc.collect()

    X_batch = df_batch[train_cols].values
    y_batch = df_batch['flag'].values

    if first_call:
        clf.partial_fit(X_batch, y_batch, classes=CLASSES)
        first_call = False
    else:
        clf.partial_fit(X_batch, y_batch)
    del X_batch, y_batch
    gc.collect()

In [40]:
df_test = pd.read_parquet('data_test_std.pq')
df_test.shape

(600000, 388)

In [41]:
y_true = df_test['flag']
y_true.shape

(600000,)

In [43]:
y_pred_binary = clf.predict(df_test[train_cols])
y_pred_binary.shape

(600000,)

In [46]:
y_pred_proba = clf.predict_proba(df_test[train_cols])
y_pred_proba.shape

(600000, 2)

In [47]:
del df_test
gc.collect()

884

In [48]:
log_reg_accuracy = accuracy_score(y_true, y_pred_binary)
log_reg_roc_auc = roc_auc_score(y_true, y_pred_proba[:, 1])

In [49]:
print(f'Accuracy = {log_reg_accuracy:.3f}, ROC AUC = {log_reg_roc_auc:.3f}')

Accuracy = 0.958, ROC AUC = 0.657


### Ручной Grid Search для SGDClassifier

In [58]:
# задаем параметры
LOSS = 'log_loss'
PENALTY = 'l2'
LIST_ALPHA = [5e-5, 1e-4, 5e-4, 1e-3, 1e-2, 1e-1, 5e-1, 6e-1, 7e-1, 1]
SEED = 42
CLASSES = [0, 1]

In [59]:
for alpha in LIST_ALPHA:
    clf = SGDClassifier(loss=LOSS, penalty=PENALTY, alpha=alpha, random_state=SEED)
    reader = pq.ParquetFile('data_train_std.pq')
    train_cols = [col for col in reader.schema.names if col not in ['id', 'flag']]
    first_call = True

    for i in range(reader.num_row_groups):
        table = reader.read_row_group(i)
        df_batch = table.to_pandas()
    
        del table
        gc.collect()

        X_batch = df_batch[train_cols].values
        y_batch = df_batch['flag'].values

        if first_call:
            clf.partial_fit(X_batch, y_batch, classes=CLASSES)
            first_call = False
        else:
            clf.partial_fit(X_batch, y_batch)
        del X_batch, y_batch
        gc.collect()

    df_test = pd.read_parquet('data_test_std.pq')
    y_true = df_test['flag']
    y_pred_binary = clf.predict(df_test[train_cols])
    y_pred_proba = clf.predict_proba(df_test[train_cols])
    del df_test
    gc.collect()
    log_reg_accuracy = accuracy_score(y_true, y_pred_binary)
    log_reg_roc_auc = roc_auc_score(y_true, y_pred_proba[:, 1])
    print(f'Learning rate = {alpha:.5f}: Accuracy = {log_reg_accuracy:.3f}, ROC AUC = {log_reg_roc_auc:.3f}')

Learning rate = 0.00005: Accuracy = 0.954, ROC AUC = 0.627
Learning rate = 0.00010: Accuracy = 0.958, ROC AUC = 0.657
Learning rate = 0.00050: Accuracy = 0.957, ROC AUC = 0.672
Learning rate = 0.00100: Accuracy = 0.959, ROC AUC = 0.663
Learning rate = 0.01000: Accuracy = 0.964, ROC AUC = 0.623
Learning rate = 0.10000: Accuracy = 0.964, ROC AUC = 0.669
Learning rate = 0.50000: Accuracy = 0.965, ROC AUC = 0.706
Learning rate = 0.60000: Accuracy = 0.965, ROC AUC = 0.706
Learning rate = 0.70000: Accuracy = 0.965, ROC AUC = 0.706
Learning rate = 1.00000: Accuracy = 0.965, ROC AUC = 0.705
